# ContextTranslate — Qwen3-4B QLoRA + dịch chương + xuất SQL Server
**Dùng Google Colab với GPU**. Notebook huấn luyện LoRA adapter Qwen3 từ các cặp Anh–Việt có metadata `speaker`, `listener`, `scene`, dịch lần lượt các chunk và xuất JSON + TXT để nhập vào SQL Server trên Windows.

⚠️ **Chưa được chạy trên GPU thực tế trong bản phát hành này.** API thư viện có thể đổi. Vui lòng dùng Colab GPU, chạy từng cell, đối chiếu bản dịch.

**Bảo vệ dữ liệu:** các ví dụ `demo_curated_pairs.csv` là mẫu **biên tập sơ bộ cho thử nghiệm**, chưa phải dữ liệu được bạn duyệt. Mode `production` chỉ train ví dụ có `human_reviewed=1`. Không đưa nội dung bản quyền lên nơi công cộng khi chưa có quyền.

## 0. Runtime → Change runtime type → GPU
Nếu Colab không cấp GPU, không chạy bước train; GPU không được bảo đảm trên Free. Trên Colab, chạy lần lượt từ trên xuống.

In [ ]:
import torch
assert torch.cuda.is_available(), "Chưa có GPU CUDA! Vào Runtime → Change runtime type → T4 GPU (nếu có)."
print("GPU:", torch.cuda.get_device_name(0))
print("GPU RAM (GB):", round(torch.cuda.get_device_properties(0).total_memory/1e9, 1))

## 1. Cài các thư viện cần thiết
Nếu sau khi cài Colab yêu cầu Restart session, hãy khởi động lại runtime và chạy lại các cell từ đầu.

In [ ]:
%pip -q install -U unsloth pandas datasets accelerate transformers peft bitsandbytes

## 2. Gắn Google Drive để giữ adapter và bản dịch sau khi Colab ngắt
Trên Drive tạo thư mục `MyDrive/ContextTranslateCloud/data`. Upload ba file từ gói ZIP: `demo_curated_pairs.csv`, `book_profile.json`, `EXCERPT_chapter_1_incomplete.txt`. Có thể dùng cell upload tiếp theo nếu muốn.

In [ ]:
from google.colab import drive, files
drive.mount('/content/drive')
from pathlib import Path
import os, json, hashlib, csv, re, time, random, shutil
import pandas as pd
ROOT=Path('/content/drive/MyDrive/ContextTranslateCloud')
DATA=ROOT/'data'; DATA.mkdir(parents=True,exist_ok=True)
MODELS=ROOT/'models'; MODELS.mkdir(parents=True,exist_ok=True)
OUTPUT=ROOT/'output'; OUTPUT.mkdir(parents=True,exist_ok=True)
print('ROOT:',ROOT)
print('Trong DATA:', [p.name for p in DATA.iterdir()])

**Nếu Drive chưa có file:** Chạy cell dưới đây rồi upload cả 3 file dữ liệu. Bạn có thể bỏ qua khi các file đã có trong Drive.

In [ ]:
# BỎ QUA CELL NÀY nếu đã upload dữ liệu vào thư mục data trên Google Drive.
# received = files.upload()
# for name, content in received.items():
#     destination = DATA / Path(name).name
#     destination.write_bytes(content)
#     print('Đã lưu:', destination)

## 3. Đọc dataset — phân biệt chế độ demo / thật
- `RUN_MODE='demo'`: lấy cặp đã được biên tập sơ bộ (`ready_for_demo=1`) để **thử quy trình**.
- `RUN_MODE='production'`: chỉ lấy dòng do bạn duyệt (`human_reviewed=1`).
- Để train bộ từ SQL Server xuất ra, đổi `CSV_NAME = 'train_pairs_from_sql.csv'` và `RUN_MODE='production'`.

Luôn kiểm tra dữ liệu trước khi train. Dữ liệu sai xưng hô có thể làm model học sai.

In [ ]:
RUN_MODE='demo'  # 'demo' hoặc 'production'
CSV_NAME='demo_curated_pairs.csv'
CSV_PATH=DATA/CSV_NAME
PROFILE_PATH=DATA/'book_profile.json'
assert CSV_PATH.exists(), f'Thiếu CSV: {CSV_PATH}'
assert PROFILE_PATH.exists(), f'Thiếu profile: {PROFILE_PATH}'
profile=json.loads(PROFILE_PATH.read_text(encoding='utf-8-sig'))
df=pd.read_csv(CSV_PATH,dtype=str,keep_default_na=False,encoding='utf-8-sig')
for col in ['source_en','target_vi','speaker','listener','scene','chapter']:
    if col not in df: df[col]=''
flag='ready_for_demo' if RUN_MODE=='demo' else 'human_reviewed'
if flag not in df:
    raise ValueError(f'CSV thiếu cột {flag}. Không train từ dữ liệu chưa được duyệt.')
df=df[df[flag].astype(str).str.strip().isin(['1','true','True','yes'])].copy()
for col in ['source_en','target_vi']:
    df[col]=df[col].str.strip()
df=df[(df.source_en!='') & (df.target_vi!='')]
# Không để dữ liệu cùng key nhưng target khác nhau trong cùng phiên train.
df=df.drop_duplicates(subset=['source_en','speaker','listener','scene'],keep='first').reset_index(drop=True)
if len(df)<8:
    raise RuntimeError(f'Chỉ có {len(df)} cặp hợp lệ. Cần >=8 để làm ví dụ train/test.')
print(f'Run mode: {RUN_MODE} | Pairs: {len(df)}')
print(df[['source_en','target_vi','speaker','listener','scene']].head(5).to_string(index=False))

## 4. Tạo prompt nhất quán với ngữ cảnh nhân vật
Cách nói `speaker/listener/scene` **phải được đưa vào prompt lúc train và lúc dịch**. Đoạn hội thoại chưa rõ người nói đặt `speaker=unknown` và `scene=mixed`, không được ép tất cả lời thoại của một chương thành giọng một người.

In [ ]:
SYSTEM=(
    'Bạn là biên dịch viên Anh–Việt chuyên light novel/fanfiction. '
    'Chỉ trả về bản dịch tiếng Việt. Bảo toàn ý, nhịp kể, tên riêng, xuống dòng; '
    'không bịa tình tiết. Cách xưng hô phụ thuộc đúng vào người nói, '
    'người nghe và trạng thái quan hệ của từng cảnh. '
    'Khi chưa đủ thông tin, không áp đặt xưng hô thù địch lên cảnh bình thường.'
)

def pronoun_notes(speaker='',listener='',scene='mixed'):
    rules=profile.get('pronoun_rules',[])
    if speaker and listener:
        rules=[r for r in rules if r['speaker']==speaker and r['listener']==listener
               and r['scene'] in (scene,'general')]
    # Đoạn nhiều nhân vật cần bản đồ quan hệ ngắn thay vì ép một cặp.
    return '\n'.join(f"{r['speaker']} -> {r['listener']} [{r['scene']}]: "
                     f"xưng {r['self_word']}, gọi {r['address_word']}"
                     for r in rules[:18]) or '(chưa có quy tắc cụ thể)'

def build_user(source_en,speaker='',listener='',scene='mixed',recent_context=''):
    return (f"Truyện: {profile.get('title','')}\n"
            f"Phong cách: {profile.get('style_guide','')}\n"
            f"Người nói: {speaker or 'chưa xác định'}\n"
            f"Người nghe: {listener or 'chưa xác định'}\n"
            f"Trạng thái cảnh: {scene or 'mixed'}\n"
            f"Quy tắc xưng hô có liên quan:\n{pronoun_notes(speaker,listener,scene)}\n"
            f"Bối cảnh đoạn trước (tham khảo, KHÔNG dịch lại):\n{recent_context or '(không có)'}\n"
            f"NỘI DUNG CẦN DỊCH:\n{source_en}")

# Validation/test tách dựa trên nguyên văn tiếng Anh để tránh rò rỉ một câu trùng.
keyed={k:g for k,g in df.groupby('source_en',sort=False)}
keys=list(keyed); random.Random(3407).shuffle(keys)
n_val=max(2,round(len(keys)*0.2))
val_keys=set(keys[:n_val])
train_rows=df[~df.source_en.isin(val_keys)].to_dict('records')
val_rows=df[df.source_en.isin(val_keys)].to_dict('records')
assert train_rows and val_rows
print('Train:',len(train_rows),'Validation:',len(val_rows))

## 5. Tải Qwen3-4B 4-bit
Model `unsloth/Qwen3-4B-unsloth-bnb-4bit` có sẵn trên Hugging Face. Colab cần Internet để tải lần đầu. `MAX_SEQ_LEN=2048` phù hợp lần thử đầu trên T4.

In [ ]:
from unsloth import FastLanguageModel
MODEL_ID='unsloth/Qwen3-4B-unsloth-bnb-4bit'
MAX_SEQ_LEN=2048
model,tokenizer=FastLanguageModel.from_pretrained(
    model_name=MODEL_ID,max_seq_length=MAX_SEQ_LEN,load_in_4bit=True,dtype=None
)
if tokenizer.pad_token is None:
    tokenizer.pad_token=tokenizer.eos_token
print('Qwen đã được nạp:',MODEL_ID)

## 6. Chạy thử model gốc trên câu ở tập validation
Lưu bản dịch ban đầu để so sánh sau train. Kết quả không đại diện chất lượng tổng thể khi tập test quá nhỏ.

In [ ]:
def generate_vi(source_en,speaker='',listener='',scene='mixed',recent_context='',max_new_tokens=512):
    from unsloth import FastLanguageModel
    FastLanguageModel.for_inference(model)
    messages=[{'role':'system','content':SYSTEM},
              {'role':'user','content':build_user(source_en,speaker,listener,scene,recent_context)}]
    inp=tokenizer.apply_chat_template(
        messages,tokenize=True,add_generation_prompt=True,enable_thinking=False,
        return_tensors='pt').to('cuda')
    if inp.shape[-1]+max_new_tokens>MAX_SEQ_LEN:
        raise ValueError(f'Chunk dài quá context window: input={inp.shape[-1]} tokens.')
    with torch.inference_mode():
        outs=model.generate(input_ids=inp,max_new_tokens=max_new_tokens,
                            do_sample=True,temperature=0.6,top_p=0.8,top_k=20,
                            pad_token_id=tokenizer.eos_token_id,
                            eos_token_id=tokenizer.eos_token_id)
    return tokenizer.decode(outs[0,inp.shape[-1]:],skip_special_tokens=True).strip()

**Đối chiếu model gốc trên một vài ví dụ validation**

In [ ]:
base_predictions=[]
for row in val_rows[:3]:
    base_predictions.append(generate_vi(row['source_en'],row['speaker'],row['listener'],row['scene']))
    print('EN:',row['source_en'][:140]); print('BEFORE:',base_predictions[-1][:250]); print('-'*25)

## 7. Tạo QLoRA adapter và dữ liệu huấn luyện
Chúng ta huấn luyện **chỉ các token của câu trả lời tiếng Việt**, còn prompt/context được mask bằng `-100`. Điều này giúp model học phản hồi thay vì học lặp lại câu hỏi. Samples quá dài được bỏ qua, không âm thầm cắt mất bản dịch.

In [ ]:
model=FastLanguageModel.get_peft_model(
    model,r=8,lora_alpha=16,lora_dropout=0,bias='none',
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    use_gradient_checkpointing='unsloth',random_state=3407
)
from datasets import Dataset
from transformers import DataCollatorForSeq2Seq

def tokenize_row(row):
    msg=[{'role':'system','content':SYSTEM},
         {'role':'user','content':build_user(row['source_en'],row['speaker'],row['listener'],row['scene'])}]
    prompt=tokenizer.apply_chat_template(msg,tokenize=True,
        add_generation_prompt=True,enable_thinking=False)
    # Chỉ giám sát phần dịch tiếng Việt, dùng EOS để kết thúc phản hồi.
    completion=tokenizer(row['target_vi'],add_special_tokens=False)['input_ids']
    completion=completion+[tokenizer.eos_token_id]
    seq=prompt+completion
    if len(seq)>MAX_SEQ_LEN:
        return None
    return {'input_ids':seq,'attention_mask':[1]*len(seq),
            'labels':[-100]*len(prompt)+completion}

train_tok=[tokenize_row(r) for r in train_rows]
val_tok=[tokenize_row(r) for r in val_rows]
train_tok=[r for r in train_tok if r is not None]
val_tok=[r for r in val_tok if r is not None]
assert train_tok and val_tok,'Dataset tokenized rỗng; hãy giảm độ dài đoạn.'
train_dataset=Dataset.from_list(train_tok)
val_dataset=Dataset.from_list(val_tok)
collator=DataCollatorForSeq2Seq(tokenizer=tokenizer,model=model,padding=True,
                               label_pad_token_id=-100,pad_to_multiple_of=8)
print('Tokenized:',len(train_dataset),'train /',len(val_dataset),'val')

## 8. Huấn luyện thử trong khoảng 20 bước
`max_steps=20` chỉ để test pipeline, chưa nhằm đạt chất lượng dịch tốt. Với dataset lớn hơn, có thể đổi sang 1–3 epochs và theo dõi validation loss. Checkpoint đặt trên Drive để có thể giữ trạng thái lâu hơn, nhưng Colab vẫn có thể ngắt bất ngờ.

In [ ]:
from transformers import Trainer, TrainingArguments
ADAPTER_DIR=MODELS/'qwen3_1p7b_contexttranslate_lora_v1'
CHECKPOINT_DIR=MODELS/'training_checkpoints_v1'
BF16=torch.cuda.is_bf16_supported()
args=TrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    per_device_train_batch_size=1,gradient_accumulation_steps=4,
    per_device_eval_batch_size=1,
    max_steps=20,learning_rate=1e-4,warmup_steps=2,
    lr_scheduler_type='linear',weight_decay=0.01,
    logging_steps=2,eval_strategy='steps',eval_steps=10,
    save_strategy='steps',save_steps=10,save_total_limit=2,
    fp16=not BF16,bf16=BF16,optim='adamw_8bit',
    gradient_checkpointing=False,
    report_to='none',seed=3407,remove_unused_columns=False
)
FastLanguageModel.for_training(model)
trainer=Trainer(model=model,args=args,
    train_dataset=train_dataset,eval_dataset=val_dataset,
    data_collator=collator)
# Nếu bị ngắt và checkpoint còn đủ, có thể dùng trainer.train(resume_from_checkpoint=True)
trainer.train()
trainer.save_model(str(ADAPTER_DIR))
tokenizer.save_pretrained(str(ADAPTER_DIR))
metadata={'base_model':MODEL_ID,'adapter_name':ADAPTER_DIR.name,
          'max_seq_len':MAX_SEQ_LEN,'train_rows':len(train_dataset),
          'val_rows':len(val_dataset),'run_mode':RUN_MODE,
          'training_steps':20}
(ADAPTER_DIR/'training_metadata.json').write_text(json.dumps(metadata,ensure_ascii=False,indent=2),encoding='utf-8')
print('Đã lưu adapter vào Drive:',ADAPTER_DIR)

## 9. So sánh sau fine-tune và tải bảng đánh giá về

In [ ]:
after_predictions=[]
for row in val_rows[:3]:
    after_predictions.append(generate_vi(row['source_en'],row['speaker'],row['listener'],row['scene']))
report=[]
for i,row in enumerate(val_rows[:3]):
    report.append({'source_en':row['source_en'],'reference_vi':row['target_vi'],
                   'base_vi':base_predictions[i],'finetuned_vi':after_predictions[i],
                   'speaker':row['speaker'],'listener':row['listener'],'scene':row['scene']})
report_path=OUTPUT/'before_vs_after.csv'
pd.DataFrame(report).to_csv(report_path,index=False,encoding='utf-8-sig')
print(pd.DataFrame(report)[['source_en','base_vi','finetuned_vi']].to_string(index=False))
print('Lưu bảng đối chiếu:',report_path)

## 10. Dịch một chương truyện sau khi đã train
Colab tiếp tục dùng model đang ở RAM. Nếu runtime đã ngắt, chạy bước cài thư viện, mount Drive và dùng **cell nạp adapter** ngay bên dưới thay vì train lại.

In [ ]:
# Khi vào Colab ở phiên mới: chạy cell 2,4,6,10,12,16 (định nghĩa generate_vi),
# KHÔNG chạy cell 14 nạp base riêng (tránh dùng GPU RAM hai lần).
# KHÔNG chạy các cell train; bật True dưới đây.
RELOAD_SAVED_ADAPTER=False
if RELOAD_SAVED_ADAPTER:
    from unsloth import FastLanguageModel
    MAX_SEQ_LEN=2048
    ADAPTER_DIR=MODELS/'qwen3_1p7b_contexttranslate_lora_v1'
    assert (ADAPTER_DIR/'adapter_config.json').exists(), f'Thiếu adapter: {ADAPTER_DIR}'
    model,tokenizer=FastLanguageModel.from_pretrained(
        model_name=str(ADAPTER_DIR),max_seq_length=MAX_SEQ_LEN,load_in_4bit=True)
    FastLanguageModel.for_inference(model)
    print('Đã nạp lại adapter:',ADAPTER_DIR)

**Tải một chương TXT (UTF-8) vào Drive.** Đổi `CHAPTER_FILE` bằng tên file thật và `CHAPTER_NUMBER` cho phù hợp. File ví dụ trích giữa chương 1 là **đoạn không đầy đủ**, chỉ để test; đừng coi đó là bản đầy đủ chương 1.

In [ ]:
# Nếu cần upload chương từ laptop:
# uploaded=files.upload()
# for name,b in uploaded.items():
#     (DATA/Path(name).name).write_bytes(b)

CHAPTER_FILE='EXCERPT_chapter_1_incomplete.txt'
CHAPTER_NUMBER=1
CHAPTER_TITLE='Chapter 1 (excerpt only)'
SCENE='mixed'
MAX_CHARS_PER_CHUNK=1000
RECENT_CONTEXT_CHARS=260
MAX_NEW_TOKENS=700
source_path=DATA/CHAPTER_FILE
assert source_path.exists(),f'Không tìm thấy {source_path}'
source_text=source_path.read_text(encoding='utf-8-sig').strip()
print('Ký tự EN:',len(source_text))

## 11. Chia chương theo đoạn văn, dịch từng chunk và lưu checkpoint
Nếu Colab ngắt sau chunk 3, chạy lại notebook với cùng adapter/chương. Các chunk có cùng hash và context sẽ được lấy từ JSON checkpoint trên Google Drive. File xuất có trạng thái `partial` trong khi đang dịch, `complete` khi xong.

In [ ]:
def split_long(text, max_chars):
    # Ưu tiên ranh giới câu / dấu câu, rồi đến khoảng trắng; tránh cắt giữa từ.
    parts=[]
    while len(text)>max_chars:
        candidates=[text.rfind(mark,0,max_chars) for mark in ['. ','? ','! ','; ', ', ']]
        cut=max(candidates)
        if cut<max_chars//2:
            cut=text.rfind(' ',0,max_chars)
        if cut<max_chars//2:
            cut=max_chars
        else:
            cut+=1
        parts.append(text[:cut].strip())
        text=text[cut:].strip()
    if text:parts.append(text)
    return parts

def split_chapter(text,max_chars=1000):
    paragraphs=re.split(r'\n\s*\n+',text.strip())
    result=[];buffer=''
    for paragraph in paragraphs:
        p=paragraph.strip()
        if not p:continue
        for piece in split_long(p,max_chars):
            combined=(buffer+'\n\n'+piece).strip() if buffer else piece
            if len(combined)>max_chars and buffer:
                result.append(buffer);buffer=piece
            else:buffer=combined
    if buffer:result.append(buffer)
    return result

def h(text):return hashlib.sha256(text.encode('utf-8')).hexdigest()
def h_json(obj):return h(json.dumps(obj,ensure_ascii=False,sort_keys=True))

chunks=split_chapter(source_text,MAX_CHARS_PER_CHUNK)
print('Chunks:',len(chunks),'lengths:',[len(c) for c in chunks])
assert chunks,'Chương không có nội dung!'

In [ ]:
MODEL_NAME='Qwen3-1.7B-QLoRA-ContextTranslate-v1'
export_path=OUTPUT/f'blue_archive_chapter_{CHAPTER_NUMBER}_translation.json'
if export_path.exists():
    old_doc=json.loads(export_path.read_text(encoding='utf-8'))
    if old_doc.get('book_key')!=profile.get('book_key') or old_doc.get('chapter_number')!=CHAPTER_NUMBER:
        raise ValueError('Checkpoint không khớp book/chapter!')
else:old_doc={}
old_chunks={x['chunk_index']:x for x in old_doc.get('chunks',[])}
doc={'format_version':1,'book_key':profile.get('book_key'),
     'chapter_number':CHAPTER_NUMBER,'chapter_title':CHAPTER_TITLE,
     'model_name':MODEL_NAME,'status':'partial','chunks':[]}
for idx,text_en in enumerate(chunks):
    # Chỉ tham chiếu phần GỐC trước đó, không lấy chương tương lai.
    prev_en=chunks[idx-1][-RECENT_CONTEXT_CHARS:] if idx>0 else ''
    c_hash=h_json({'profile':profile,'scene':SCENE,'previous_source':prev_en,
                   'model_name':MODEL_NAME,'prompt_version':'v1'})
    src_hash=h(text_en)
    old=old_chunks.get(idx)
    if old and old.get('source_hash')==src_hash and old.get('context_hash')==c_hash and old.get('target_vi'):
        record=old
        print(f'Chunk {idx+1}/{len(chunks)}: cache')
    else:
        # Trên văn bản dài/nhiều nhân vật, không gán speaker=Yuuka cho cả chunk.
        vi=generate_vi(text_en,scene=SCENE,recent_context=prev_en,max_new_tokens=MAX_NEW_TOKENS)
        if not vi.strip():
            raise RuntimeError(f'Chunk {idx} dịch ra kết quả rỗng. Kiểm tra trước khi chạy tiếp.')
        record={'chunk_index':idx,'source_en':text_en,'target_vi':vi,
                'source_hash':src_hash,'context_hash':c_hash,'model_name':MODEL_NAME,
                'input_tokens':None,'output_tokens':None}
        print(f'Chunk {idx+1}/{len(chunks)}: đã dịch')
    doc['chunks'].append(record)
    doc['status']='complete' if len(doc['chunks'])==len(chunks) else 'partial'
    tmp=export_path.with_suffix('.tmp')
    tmp.write_text(json.dumps(doc,ensure_ascii=False,indent=2),encoding='utf-8')
    os.replace(tmp,export_path)

txt_path=OUTPUT/f'blue_archive_chapter_{CHAPTER_NUMBER}_translation.txt'
txt_path.write_text('\n\n'.join(x['target_vi'] for x in doc['chunks']),encoding='utf-8')
print('JSON:',export_path)
print('TXT:',txt_path)
print('Status:',doc['status'])

## 12. Download hai file trên về Windows rồi nhập SQL Server
`files.download()` chỉ tải xuống laptop; còn adapter tiếp tục nằm ở Google Drive. Khi bắt đầu chương mới, **không cần train lại**: nạp adapter đã lưu và chỉ chạy phần dịch chương.

In [ ]:
from google.colab import files
files.download(str(export_path))
files.download(str(txt_path))

## 13. Về laptop: nhập JSON vào SQL Server
Trong ZIP đã có đủ các file Windows tại thư mục `windows/`. **Không cần copy file sang dự án cũ.**

Mở PowerShell tại thư mục `ContextTranslate_Qwen_Colab_SQLServer/windows` (đã cài thư viện và cấu hình `.env` theo README):

```powershell
.\.venv\Scripts\python.exe import_chapter_to_sql.py ".\blue_archive_chapter_1_translation.json" --dry-run
.\.venv\Scripts\python.exe import_chapter_to_sql.py ".\blue_archive_chapter_1_translation.json"
.\.venv\Scripts\python.exe read_chapter.py --chapter 1
```

Nếu file ở `Downloads`, thay đường dẫn JSON bằng `"$env:USERPROFILE\Downloads\blue_archive_chapter_1_translation.json"`.

SQL Server (SSMS):

```sql
USE ContextTranslateDB;
SELECT c.ChapterNumber, t.ChunkIndex, t.TargetVI
FROM dbo.TranslationChunks AS t
JOIN dbo.Chapters AS c ON t.ChapterId=c.ChapterId
JOIN dbo.Books AS b ON b.BookId=c.BookId
WHERE b.BookKey=N'blue-archive' AND c.ChapterNumber=1
ORDER BY t.ChunkIndex;
```

